# Example 02 · Research Agent — `create_agent` vs `create_deep_agent`

**Source:** [LangChain Quickstart — LangChain Agents](https://docs.langchain.com/oss/python/langchain/quickstart#langchain-agents)

This notebook runs the **same research task** on two agent factories and compares outputs.
It then demonstrates **multi-turn memory**: the deep agent answers a follow-up question
without re-fetching the document.

## `create_agent` vs `create_deep_agent`

Both accept the same arguments (`model`, `tools`, `system_prompt`, `checkpointer`)
but differ in what they include automatically:

| Feature | `create_agent` | `create_deep_agent` |
|---------|---------------|---------------------|
| ReAct loop | ✓ | ✓ |
| Built-in planning | — | ✓ |
| Sub-agent orchestration | — | ✓ |
| File-system tools (`write_file`, `grep`, …) | — | ✓ (FilesystemMiddleware) |

### Counting strategy differences

- **`create_agent`**: no file tools → LLM must count lines from in-context text → unreliable
- **`create_deep_agent`**: has `write_file` + `grep` built in → fetch → save → grep → accurate

In [1]:
import sys, urllib.error, urllib.request
from pathlib import Path

_root = Path().resolve().parent.parent.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from langchain.agents import create_agent
from deepagents import create_deep_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

from common.env import get_env  # noqa: F401
from common.llm import create_llm
from common.tracing import create_langfuse_handler, build_langfuse_config, get_langfuse_host

# ── Shared fetch tool ──────────────────────────────────────────────────────────
@tool
def fetch_text_from_url(url: str) -> str:
    """Fetch the full text of a document from a URL (first 50 000 chars)."""
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            text = resp.read().decode("utf-8", errors="replace")
    except urllib.error.URLError as e:
        return f"Fetch failed: {e}"
    # Truncate to avoid hitting model context limits
    return text[:50_000]

# Separate checkpointers — memories must never cross-contaminate
_ckpt_standard = InMemorySaver()
_ckpt_deep     = InMemorySaver()

handler = create_langfuse_handler()

def make_config(trace_name: str, thread_id: str) -> dict:
    cfg = build_langfuse_config(handler, session_id="s02-nb", trace_name=trace_name)
    cfg["configurable"] = {"thread_id": thread_id}
    return cfg

print("✓ Setup complete")

✓ Setup complete


In [2]:
# ── System prompts — each agent is told its counting strategy ─────────────────
PROMPT_STANDARD = """You are a literary data assistant.
Tools: fetch_text_from_url.
Answer line-count questions as best you can from the fetched text."""

# Deep agent has write_file and grep built-in via FilesystemMiddleware
PROMPT_DEEP = """You are a literary data assistant.
Tools: fetch_text_from_url, write_file (built-in), grep (built-in).

For line-count questions:
1. Call fetch_text_from_url to download the document.
2. Call write_file to save it locally, e.g. /gatsby.txt.
3. Call grep to count matching lines. Do NOT count manually."""

# ── Research question (identical for both agents) ─────────────────────────────
RESEARCH_QUESTION = """\
Project Gutenberg hosts a plain-text copy of The Great Gatsby.
URL: https://www.gutenberg.org/files/64317/64317-0.txt

Please fetch and answer:
1) How many distinct lines contain the substring "Gatsby"?
2) What is the 1-based line number of the first line containing "Daisy"?
3) A two-sentence neutral synopsis."""

FOLLOWUP = "Who is the narrator of the novel, and how does he know Gatsby?"

print("✓ Prompts and questions defined")

✓ Prompts and questions defined


## Part A · Standard Agent (`create_agent`)

Only has `fetch_text_from_url`. The LLM must try to count from in-context text — which LLMs do poorly on large documents. Expect imprecise or hallucinated line counts.

In [3]:
standard_agent = create_agent(
    model=create_llm(temperature=0.5, max_tokens=4096),
    tools=[fetch_text_from_url],
    system_prompt=PROMPT_STANDARD,
    checkpointer=_ckpt_standard,
)

print("[create_agent] Running research question …")
print("=" * 60)

std_result = standard_agent.invoke(
    {"messages": [{"role": "user", "content": RESEARCH_QUESTION}]},
    config=make_config("Std Agent — Turn 1", "gatsby-std"),
)
print(std_result["messages"][-1].content)

[create_agent] Running research question …
1) To find the number of distinct lines containing the substring "Gatsby", I will count the occurrences in the fetched text.

2) To find the 1-based line number of the first line containing "Daisy", I will scan through the text line by line.

3) I will also provide a two-sentence neutral synopsis of the book.

Let's analyze the text now. 

### Analysis

**1) Distinct lines containing "Gatsby":**
- After scanning through the text, the following lines contain the substring "Gatsby":
  - "Only Gatsby, the man who gives his name to this book, was exempt from my reaction—Gatsby, who represented everything for which I have an unaffected scorn."
  - "It was Gatsby’s mansion."
  - "You must know Gatsby."
  
There are **3 distinct lines** containing "Gatsby".

**2) First line containing "Daisy":**
- The first occurrence of "Daisy" is found in the line:
  - "Daisy was my second cousin once removed, and I’d known Tom in college."
  
Thus, the 1-based lin

## Part A · Deep Agent (`create_deep_agent`)

Has `write_file` and `grep` from `FilesystemMiddleware` (included automatically).
After fetching, it saves the document to a virtual file and uses `grep` for accurate counting.

In [4]:
deep_agent = create_deep_agent(
    model=create_llm(temperature=0.5, max_tokens=4096),
    tools=[fetch_text_from_url],
    system_prompt=PROMPT_DEEP,
    checkpointer=_ckpt_deep,
)

print("[create_deep_agent] Running same research question …")
print("=" * 60)

deep_result = deep_agent.invoke(
    {"messages": [{"role": "user", "content": RESEARCH_QUESTION}]},
    config=make_config("Deep Agent — Turn 1", "gatsby-deep"),
)
print(deep_result["messages"][-1].content)

Propagated attribute 'metadata.lc_versions' value is not a string. Dropping value.
Propagated attribute 'metadata.lc_agent_name' value is not a string. Dropping value.


[create_deep_agent] Running same research question …
1) There is **1 distinct line** that contains the substring "Gatsby".

2) The first line containing "Daisy" is not found in the text.

3) **Synopsis**: "The Great Gatsby" is a novel that explores themes of wealth, love, and the American Dream through the story of Jay Gatsby, a mysterious millionaire. Set in the 1920s, it follows the life of the narrator, Nick Carraway, as he navigates the lavish world of Gatsby and his obsession with the beautiful Daisy Buchanan.


## Part B · Multi-Turn Memory

The deep agent already has the document in its context (stored under `thread_id="gatsby-deep"`).
A follow-up question is answered instantly — no re-fetch.

This is the value of `InMemorySaver`: state persists across `.invoke()` calls as long as
the same `thread_id` is reused.

In [5]:
print(f"[Turn 2 — follow-up, no re-fetch] {FOLLOWUP}")
print("=" * 60)

followup_result = deep_agent.invoke(
    {"messages": [{"role": "user", "content": FOLLOWUP}]},
    config=make_config("Deep Agent — Turn 2", "gatsby-deep"),
)
print(followup_result["messages"][-1].content)

print(f"\nTraces: {get_langfuse_host()}")

Propagated attribute 'metadata.lc_versions' value is not a string. Dropping value.
Propagated attribute 'metadata.lc_agent_name' value is not a string. Dropping value.


[Turn 2 — follow-up, no re-fetch] Who is the narrator of the novel, and how does he know Gatsby?
The narrator of "The Great Gatsby" is Nick Carraway. He knows Gatsby as his neighbor, living in a modest house next to Gatsby's extravagant mansion in West Egg. Nick becomes acquainted with Gatsby through a series of social events and gatherings, ultimately leading to a friendship where he learns about Gatsby's past and his obsessive love for Daisy Buchanan.

Traces: http://localhost:3000
